#System Capacity & Care Load Analytics for Unaccompanied Children

Detailed guide and project requirements for the System Capacity & Care Load Analytics for Unaccompanied Children analysis.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Cognifyz_project/HHS_Unaccompanied_Alien_Children_Program.csv')

df.columns = df.columns.str.lower().str.replace('*', '', regex=False).str.replace(' ', '_')

df['date'] = pd.to_datetime(df['date'], errors='coerce')

numeric_cols = ['children_apprehended_and_placed_in_cbp_custody', 'children_in_cbp_custody',
                'children_transferred_out_of_cbp_custody', 'children_in_hhs_care',
                'children_discharged_from_hhs_care']
for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df = df.dropna(how='all').sort_values('date', ascending=False).reset_index(drop=True)

print(df.head())
print(df.info())
print(df.describe())

        date  children_apprehended_and_placed_in_cbp_custody  \
0 2025-12-21                                               6   
1 2025-12-18                                              11   
2 2025-12-17                                               7   
3 2025-12-16                                               8   
4 2025-12-15                                              11   

   children_in_cbp_custody  children_transferred_out_of_cbp_custody  \
0                       18                                       11   
1                       50                                        6   
2                       31                                       11   
3                       54                                       15   
4                       42                                        9   

   children_in_hhs_care  children_discharged_from_hhs_care  
0                  2484                                 14  
1                  2472                                 16  
2    

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Cognifyz_project/HHS_Unaccompanied_Alien_Children_Program.csv')

df.columns = df.columns.str.lower().str.replace('*', '', regex=False).str.replace(' ', '_')

df['date'] = pd.to_datetime(df['date'], errors='coerce')

numeric_cols = ['children_apprehended_and_placed_in_cbp_custody', 'children_in_cbp_custody',
                'children_transferred_out_of_cbp_custody', 'children_in_hhs_care',
                'children_discharged_from_hhs_care']
for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df = df.dropna(how='all').sort_values('date', ascending=False).reset_index(drop=True)

print(df.head())
print(df.info())
print(df.describe())

        date  children_apprehended_and_placed_in_cbp_custody  \
0 2025-12-21                                               6   
1 2025-12-18                                              11   
2 2025-12-17                                               7   
3 2025-12-16                                               8   
4 2025-12-15                                              11   

   children_in_cbp_custody  children_transferred_out_of_cbp_custody  \
0                       18                                       11   
1                       50                                        6   
2                       31                                       11   
3                       54                                       15   
4                       42                                        9   

   children_in_hhs_care  children_discharged_from_hhs_care  
0                  2484                                 14  
1                  2472                                 16  
2    

In [ ]:
import os


df_agg = df.groupby('date')[numeric_cols].sum().reset_index()

date_range = pd.date_range(start=df_agg['date'].min(), end=df_agg['date'].max(), freq='D')
df = df_agg.set_index('date').reindex(date_range).reset_index().rename(columns={'index': 'date'})

df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both').fillna(0).astype(int)

if not os.path.exists('data'):
    os.makedirs('data')

df.to_csv('data/uac_clean.csv', index=False)

In [ ]:
import numpy as np

def validate_data(df):
    anomalies = {}

    # Duplicates
    duplicates = df[df.duplicated(subset=['date'], keep=False)]
    if not duplicates.empty:
        anomalies['duplicates'] = duplicates

    missing_dates = df['date'].isna().sum()
    anomalies['missing_dates'] = missing_dates

    # Logical constraints
    df['invalid_transfers'] = df['children_transferred_out_of_cbp_custody'] > df['children_in_cbp_custody']
    df['invalid_discharges'] = df['children_discharged_from_hhs_care'] > df['children_in_hhs_care']
    anomalies['invalid_rows'] = df[df['invalid_transfers'] | df['invalid_discharges']]

    # Negatives
    negatives = (df[numeric_cols] < 0).any(axis=1)
    anomalies['negatives'] = df[negatives]

    return anomalies

# Usage in notebook
anomalies = validate_data(df)
for key, val in anomalies.items():
    if isinstance(val, (int, np.integer)):
        print(f"{key}: {val} issues")
    else:
        print(f"{key}: {len(val)} issues")
        if not val.empty:
            print(val.head())

missing_dates: 0 issues
invalid_rows: 121 issues
         date  children_apprehended_and_placed_in_cbp_custody  \
12 2023-01-24                                              47   
13 2023-01-25                                              20   
14 2023-01-26                                              20   
21 2023-02-02                                              15   
22 2023-02-03                                              15   

    children_in_cbp_custody  children_transferred_out_of_cbp_custody  \
12                       42                                       47   
13                       22                                       41   
14                       27                                       33   
21                       13                                       23   
22                       14                                       20   

    children_in_hhs_care  children_discharged_from_hhs_care  \
12                  7433                                175   
1

#Phase 2: Data Quality & Validation

In [ ]:
def validate_data(df):
    anomalies = {}

    # Duplicates
    duplicates = df[df.duplicated(subset=['date'], keep=False)]
    if not duplicates.empty:
        anomalies['duplicates'] = duplicates

    missing_dates = df['date'].isna().sum()
    anomalies['missing_dates'] = missing_dates

    # Logical constraints
    df['invalid_transfers'] = df['children_transferred_out_of_cbp_custody'] > df['children_in_cbp_custody']
    df['invalid_discharges'] = df['children_discharged_from_hhs_care'] > df['children_in_hhs_care']
    anomalies['invalid_rows'] = df[df['invalid_transfers'] | df['invalid_discharges']]

    # Negatives
    negatives = (df[numeric_cols] < 0).any(axis=1)
    anomalies['negatives'] = df[negatives]

    return anomalies

# Usage in notebook
anomalies = validate_data(df)
for key, val in anomalies.items():
    if isinstance(val, (int, np.integer)):
        print(f"{key}: {val} issues")
    else:
        print(f"{key}: {len(val)} issues")
        if not val.empty:
            print(val.head())

missing_dates: 0 issues
invalid_rows: 121 issues
         date  children_apprehended_and_placed_in_cbp_custody  \
12 2023-01-24                                              47   
13 2023-01-25                                              20   
14 2023-01-26                                              20   
21 2023-02-02                                              15   
22 2023-02-03                                              15   

    children_in_cbp_custody  children_transferred_out_of_cbp_custody  \
12                       42                                       47   
13                       22                                       41   
14                       27                                       33   
21                       13                                       23   
22                       14                                       20   

    children_in_hhs_care  children_discharged_from_hhs_care  \
12                  7433                                175   
1

In [ ]:
df.loc[df['invalid_transfers'], 'children_transferred_out_of_cbp_custody'] = df['children_in_cbp_custody']


anomalies = validate_data(df)
print("--- Validation after correction ---")
for key, val in anomalies.items():
    if isinstance(val, (int, np.integer)):
        print(f"{key}: {val} issues")
    else:
        print(f"{key}: {len(val)} issues")
        if not val.empty:
            print(val.head())

--- Validation after correction ---
missing_dates: 0 issues
invalid_rows: 0 issues
negatives: 0 issues


In [ ]:
def compute_metrics(df):
    df['total_system_load'] = df['children_in_cbp_custody'] + df['children_in_hhs_care']
    df['net_daily_intake'] = df['children_transferred_out_of_cbp_custody'] - df['children_discharged_from_hhs_care']
    df['care_load_growth_rate'] = df['children_in_hhs_care'].pct_change() * 100
    df['backlog_indicator'] = df['net_daily_intake'].clip(lower=0).rolling(window=7).sum()  # sustained positive

    return df

df = compute_metrics(df)

KPIs (from project table)
Total Children Under Care = Total System Load (avg/peak)
Net Intake Pressure = Avg net_daily_intake
Care Load Volatility Index = Std dev of care_load_growth_rate
Backlog Accumulation Rate = Avg backlog_indicator
Discharge Offset Ratio = Discharges / Transfers (ability to relieve)

In [ ]:
kpis = {
    'total_children_under_care': df['total_system_load'].mean(),
    'net_intake_pressure': df['net_daily_intake'].mean(),
    'care_load_volatility_index': df['care_load_growth_rate'].std(),
    'backlog_accumulation_rate': df['backlog_indicator'].mean(),
    'discharge_offset_ratio': df['children_discharged_from_hhs_care'].sum() / df['children_transferred_out_of_cbp_custody'].sum()
}

kpi_df = pd.DataFrame.from_dict(kpis, orient='index', columns=['Value'])
print(kpi_df)

                                  Value
total_children_under_care   6231.535814
net_intake_pressure          -54.771163
care_load_volatility_index     1.079895
backlog_accumulation_rate     75.100094
discharge_offset_ratio         1.428538


Pressure/Stress

In [ ]:
df['rolling_avg_hhs'] = df['children_in_hhs_care'].rolling(7).mean()
df['volatility'] = df['children_in_hhs_care'].rolling(14).std()


df['strain_period'] = (df['backlog_indicator'] > 50) & (df['volatility'] > 20)

print(df[df['strain_period']].head())

          date  children_apprehended_and_placed_in_cbp_custody  \
166 2023-06-27                                             154   
167 2023-06-28                                              91   
168 2023-06-29                                              81   
169 2023-06-30                                             123   
170 2023-07-01                                             166   

     children_in_cbp_custody  children_transferred_out_of_cbp_custody  \
166                      225                                      220   
167                      199                                      172   
168                      213                                      153   
169                      237                                      163   
170                      261                                      173   

     children_in_hhs_care  children_discharged_from_hhs_care  \
166                  5917                                155   
167                  5999           

#Phase 5: Streamlit Web Application

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px

st.title('UAC Care Load Analytics Dashboard')

# Load data
df = pd.read_csv('data/uac_clean.csv')
df = compute_metrics(df)
df['date'] = pd.to_datetime(df['date'])

# User capabilities
date_range = st.date_input('Select Date Range', [df['date'].min(), df['date'].max()])
granularity = st.selectbox('Time Granularity', ['Daily', 'Weekly', 'Monthly'])

# Filter
mask = (df['date'] >= pd.to_datetime(date_range[0])) & (df['date'] <= pd.to_datetime(date_range[1]))
df_filtered = df[mask]

if granularity == 'Weekly':
    df_filtered = df_filtered.resample('W', on='date').mean().reset_index()

# System Load Overview
st.subheader('System Load Overview')
fig_load = px.line(df_filtered, x='date', y='total_system_load')
st.plotly_chart(fig_load)

# CBP vs HHS Comparison
st.subheader('CBP vs HHS Load')
fig_compare = px.bar(df_filtered, x='date', y=['children_in_cbp_custody', 'children_in_hhs_care'], barmode='stack')
st.plotly_chart(fig_compare)

# Net Intake & Backlog
st.subheader('Net Intake & Backlog Trends')
fig_intake = px.area(df_filtered, x='date', y='net_daily_intake')
st.plotly_chart(fig_intake)

# KPI Cards
st.subheader('Key Performance Indicators')
kpis = {  # compute as above
    'Total Under Care': df_filtered['total_system_load'].mean()
    # ... add others
}
cols = st.columns(len(kpis))
for i, (name, val) in enumerate(kpis.items()):
    cols[i].metric(name, f"{val:.2f}")

2026-04-22 11:56:56.256 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.258 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.260 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.271 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.272 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.274 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-22 11:56:56.275 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
import os
import pandas as pd


df = pd.read_csv('/content/drive/MyDrive/Cognifyz_project/HHS_Unaccompanied_Alien_Children_Program.csv')


df.columns = df.columns.str.lower().str.replace('*', '', regex=False).str.replace(' ', '_')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

numeric_cols = [
    'children_apprehended_and_placed_in_cbp_custody',
    'children_in_cbp_custody',
    'children_transferred_out_of_cbp_custody',
    'children_in_hhs_care',
    'children_discharged_from_hhs_care'
]
for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df = df.dropna(how='all').sort_values('date').reset_index(drop=True)

# Step 3: Resample to fill date gaps
df_agg = df.groupby('date')[numeric_cols].sum().reset_index()
date_range = pd.date_range(start=df_agg['date'].min(), end=df_agg['date'].max(), freq='D')
df = df_agg.set_index('date').reindex(date_range).reset_index().rename(columns={'index': 'date'})
df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both').fillna(0).astype(int)

# Step 4: Save cleaned CSV ✅ This is what app.py needs
os.makedirs('/content/data', exist_ok=True)
df.to_csv('/content/data/uac_clean.csv', index=False)

print("✅ uac_clean.csv created successfully at /content/data/")
print(f"   Shape: {df.shape}")
print(df.head(3))

✅ uac_clean.csv created successfully at /content/data/
   Shape: (1075, 6)
        date  children_apprehended_and_placed_in_cbp_custody  \
0 2023-01-12                                              33   
1 2023-01-13                                              32   
2 2023-01-14                                              32   

   children_in_cbp_custody  children_transferred_out_of_cbp_custody  \
0                       53                                       34   
1                       52                                       34   
2                       52                                       35   

   children_in_hhs_care  children_discharged_from_hhs_care  
0                  6566                                436  
1                  6621                                415  
2                  6677                                394  


In [ ]:
import pandas as pd

df = pd.read_csv('/content/data/uac_clean.csv')

In [ ]:
app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import os

st.title('UAC Care Load Analytics Dashboard')

# Load data
df = pd.read_csv('/content/data/uac_clean.csv')
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print("✅ app.py updated successfully")

✅ app.py updated successfully


In [ ]:
! pip install streamlit -q

In [ ]:
!wget -  -o - ipv4.icanhazip.com

--2026-04-26 22:03:38--  http://-/
Resolving - (-)... failed: Name or service not known.
wget: unable to resolve host address ‘-’
--2026-04-26 22:03:38--  http://ipv4.icanhazip.com/
Resolving ipv4.icanhazip.com (ipv4.icanhazip.com)... 104.16.184.241, 104.16.185.241
Connecting to ipv4.icanhazip.com (ipv4.icanhazip.com)|104.16.184.241|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13 [text/plain]
Saving to: ‘index.html’

     0K                                                       100% 1.40M=0s

2026-04-26 22:03:38 (1.40 MB/s) - ‘index.html’ saved [13/13]

FINISHED --2026-04-26 22:03:38--
Total wall clock time: 0.06s
Downloaded: 1 files, 13 in 0s (1.40 MB/s)


In [ ]:
# Reinstall streamlit to ensure it's available in the current environment
! pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 87.8 MB/s eta 0:00:00


In [ ]:
! python -m streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.227.60.49:8501

  Stopping...
^C


In [ ]:
streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import os

st.title('UAC Care Load Analytics Dashboard')

# Load data
df = pd.read_csv('/content/data/uac_clean.csv')

def compute_metrics(df):
    df['total_system_load'] = df['children_in_cbp_custody'] + df['children_in_hhs_care']
    df['net_daily_intake'] = df['children_transferred_out_of_cbp_custody'] - df['children_discharged_from_hhs_care']
    df['care_load_growth_rate'] = df['children_in_hhs_care'].pct_change() * 100
    df['backlog_indicator'] = df['net_daily_intake'].clip(lower=0).rolling(window=7).sum()  # sustained positive
    return df

df = compute_metrics(df)
df['date'] = pd.to_datetime(df['date'])

# User capabilities
date_range = st.date_input('Select Date Range', [df['date'].min(), df['date'].max()])
granularity = st.selectbox('Time Granularity', ['Daily', 'Weekly', 'Monthly'])

# Filter
mask = (df['date'] >= pd.to_datetime(date_range[0])) & (df['date'] <= pd.to_datetime(date_range[1]))
df_filtered = df[mask]

if granularity == 'Weekly':
    df_filtered = df_filtered.resample('W', on='date').mean().reset_index()

# System Load Overview
st.subheader('System Load Overview')
fig_load = px.line(df_filtered, x='date', y='total_system_load')
st.plotly_chart(fig_load)

# CBP vs HHS Comparison
st.subheader('CBP vs HHS Load')
fig_compare = px.bar(df_filtered, x='date', y=['children_in_cbp_custody', 'children_in_hhs_care'], barmode='stack')
st.plotly_chart(fig_compare)

# Net Intake & Backlog
st.subheader('Net Intake & Backlog Trends')
fig_intake = px.area(df_filtered, x='date', y='net_daily_intake')
st.plotly_chart(fig_intake)

# KPI Cards
st.subheader('Key Performance Indicators')
kpis = {  # compute as above
    'Total Under Care': df_filtered['total_system_load'].mean()
    # ... add others
}
cols = st.columns(len(kpis))
for i, (name, val) in enumerate(kpis.items()):
    cols[i].metric(name, f"{val:.2f}")
'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

print('Streamlit app regenerated to app.py with full dashboard code')


Streamlit app regenerated to app.py with full dashboard code


In [ ]:
import os

log_file_path = 'streamlit.log'

if os.path.exists(log_file_path):
    with open(log_file_path, 'r') as f:
        log_content = f.read()
    print(log_content)
else:
    print(f"Log file not found at: {log_file_path}")




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.227.60.49:8501

  Stopping...



In [ ]:
# Install ngrok
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok


ngrok.kill()
YOUR_AUTHTOKEN = "3CuibFxTXzAgKi7nGLhw5EDopVS_32BL4r9xykoNmt5Px1KYi"

NGROK_AUTH_TOKEN = "3CuibFxTXzAgKi7nGLhw5EDopVS_32BL4r9xykoNmt5Px1KYi"

if NGROK_AUTH_TOKEN == "3CuibFxTXzAgKi7nGLhw5EDopVS_32BL4r9xykoNmt5Px1KYi":
    print("Please replace '3CuibFxTXzAgKi7nGLhw5EDopVS_32BL4r9xykoNmt5Px1KYi' with your actual ngrok authtoken from ngrok.com")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)


    !nohup python -m streamlit run app.py > streamlit_ngrok.log 2>&1 &


    public_url = ngrok.connect(8501)
    print(f"Streamlit app is available at: {public_url}")
    print("Check streamlit_ngrok.log for Streamlit output.")

Please replace '3CuibFxTXzAgKi7nGLhw5EDopVS_32BL4r9xykoNmt5Px1KYi' with your actual ngrok authtoken from ngrok.com


In [ ]:
# Run Streamlit in the background and then start localtunnel
!nohup python -m streamlit run app.py > streamlit.log 2>&1 & \
yes | npx localtunnel --port 8501


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://violet-crabs-dress.loca.lt
^C
